# Validación y curación del dataset de demo

Evaluación del AI Engine con las funciones del PR #38 y registro de entradas, respuestas, fecha y versión para revisión editorial.

**Referencia:** PR #38, commit `d62dd02ce1a358824b72336a6e4150be1ab81262`. La evidencia conservada procede del snapshot del PR #38. Las nuevas ejecuciones importan `ai-engine/` del repositorio, o el checkout indicado por `ANDREA_ENGINE_DIR`.

**Niveles de evaluación:**
1. Generador directo: análisis y funciones de generación.
2. AI Engine local: endpoint batch, selección y LangGraph.
3. Backend integrado: solicitud HTTP al despliegue de prueba.

Formatos prioritarios: LinkedIn y newsletter. FAQ sujeto al alcance confirmado. Las llamadas utilizan la cuota del proyecto configurado.


## Estado de la evidencia

Las salidas guardadas corresponden al código del PR #38, commit `d62dd02ce1a358824b72336a6e4150be1ab81262`, ejecutado desde el paquete de evaluación original. Se conserva la evidencia original sin reejecutar Gemini. La ruta de importación se adapta aquí a la estructura del repositorio.

En las dos ejecuciones, las tres solicitudes locales devolvieron HTTP 200 y generaron LinkedIn. El candidato DEMO-003, previsto para newsletter, también fue seleccionado como LinkedIn. La evaluación directa sí prueba la función newsletter de forma aislada. La cobertura automática de newsletter y la prueba del backend integrado quedan pendientes.

Dependencia: PR #38. Para reproducir antes de su integración, `ANDREA_ENGINE_DIR` puede apuntar a un checkout separado de ese PR.


## 0. Configuración de acceso

La API key se obtiene en [Google AI Studio — API Keys](https://aistudio.google.com/api-keys), mediante **Create API key**, asociada al proyecto de prueba. Los proyectos existentes se incorporan desde **Dashboard → Projects → Import projects**.

La plantilla `.env.example` define `GEMINI_API_KEY`, `GEMINI_MODEL` y las variables opcionales del backend. La configuración local se carga desde `.env`; el modelo predeterminado es `gemini-3.5-flash-lite`.

Referencia: [Autenticación de Gemini](https://ai.google.dev/gemini-api/docs/api-key).


In [1]:
from pathlib import Path
import os, sys, json, inspect
from dataclasses import asdict
from IPython.display import display, Markdown
from dotenv import load_dotenv

BASE = Path.cwd()
if not (BASE / 'curacion.py').exists():
    raise RuntimeError('Inicia Jupyter desde la carpeta curacion_demo, junto a curacion.py.')
if not (BASE / '.env').exists():
    raise FileNotFoundError('Copia .env.example a .env e introduce tu clave en el editor local.')
load_dotenv(BASE / '.env', override=True)
key = os.environ.get('GEMINI_API_KEY', '').strip()
if not key or key == 'PEGA_AQUI_TU_LLAVE':
    raise ValueError('Falta introducir una clave válida en .env. No la pegues en el notebook.')
del key
MODELO = os.environ.get('GEMINI_MODEL', 'gemini-3.5-flash-lite')
ENGINE = Path(os.environ.get('ANDREA_ENGINE_DIR') or BASE.parents[1] / 'ai-engine').resolve()
print('Entorno cargado. Modelo:', MODELO)
print('Código fuente:', ENGINE)

Entorno cargado. Modelo: gemini-3.5-flash-lite
Código fuente: C:\Users\saul_\Documents\Codex\2026-10-17\review-pr-23-in-no-country\outputs\curacion_demo\andrea_pr38


## 1. Funciones y ejemplos de referencia

Se importan los módulos del motor seleccionado y se muestra la implementación efectiva de los generadores, el enrutador y los ejemplos few-shot. El kernel debe utilizar una única versión del motor por sesión.


In [2]:
from curacion import (Registro, ClienteRegistrado, cargar_modulos, version_fuente,
                      validar_lote, guardar_json, probar_motor, probar_backend)
mod = cargar_modulos(ENGINE)
from google import genai
cliente_real = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
print('Generadores importados de:', mod.generators.__file__)
for funcion in (mod.generators.generate_linkedin, mod.generators.generate_newsletter,
                mod.graph.decide_route):
    print(inspect.getsource(funcion))
print('Ejemplos LinkedIn:')
display(mod.few_shot.LINKEDIN_EXAMPLES)
print('Ejemplos newsletter:')
display(mod.few_shot.NEWSLETTER_EXAMPLES)

Generadores importados de: C:\Users\saul_\Documents\Codex\2026-10-17\review-pr-23-in-no-country\outputs\curacion_demo\andrea_pr38\generators.py
def generate_linkedin(
    client: genai.Client,
    interaction: Interaction,
    analysis: AnalysisResult,
    model: str,
) -> str:
    """Genera una publicación de LinkedIn a partir de una interacción analizada."""

    prompt = f"""
Eres un asistente de marketing para una comunidad educativa de tecnología.

Convierte la siguiente interacción en una publicación breve para LinkedIn.

INTERACCIÓN:
Canal: {interaction.canal}
Mensaje: {interaction.texto}

ANÁLISIS:
Sentimiento: {analysis.sentimiento}
Tema: {analysis.tema}
Tipo: {analysis.tipo}
Relevancia: {analysis.relevancia}
Insight: {analysis.insight}

EJEMPLOS DE REFERENCIA (muestran la estructura y el tono esperados; no copies sus hechos ni su contenido):
{format_examples(LINKEDIN_EXAMPLES, "Publicación esperada")}

INSTRUCCIONES:
- Escribe en español.
- Usa únicamente los hechos del mensa

[{'mensaje': '¡Gente, pasé mi primera entrevista técnica y me contrataron como Trainee Backend! El proyecto de Spring Boot que hicimos en grupo fue clave.',
  'salida': '¡El esfuerzo enfocado da resultados!\n\nUna persona de nuestra comunidad acaba de asegurar su primera posición como Trainee Backend. El trabajo colaborativo y la práctica en proyectos de Spring Boot fueron decisivos para superar la entrevista técnica.\n\nLas oportunidades surgen al combinar constancia con aprendizaje aplicado. ¡Felicidades por este logro!\n\nÚnete a nuestra comunidad en Discord.\n\n#TalentoTech #ComunidadDev #DesarrolloBackend'},
 {'mensaje': 'Logré sacar mi certificación de Oracle Cloud Always Free después de un mes de estudiar. ¡Gracias por los apuntes que pasaron por el canal!',
  'salida': 'Un mes de preparación y una nueva certificación alcanzada.\n\nCelebramos que una persona de la comunidad certificó sus conocimientos en Oracle Cloud tras un mes de estudio constante.\n\nLos apuntes compartidos e

Ejemplos newsletter:


[{'mensaje': 'Lanzamos la primera versión de la herramienta de testing para la API. Ya está el repositorio disponible para quien quiera probarlo.',
  'salida': 'La comunidad publicó la versión inicial de la herramienta de pruebas para la API. El repositorio ya se encuentra disponible para revisión y pruebas colaborativas.'}]

## 2. Modelo y versión probada

Consulta de modelos disponibles en el proyecto, configuración explícita de `MODEL_NAME` y registro de procedencia del código, dependencias y fecha de ejecución.


In [3]:
disponibles = {m.name.removeprefix('models/') for m in cliente_real.models.list() if m.name}
print('Modelos Flash/Lite visibles:', sorted(n for n in disponibles if 'flash' in n or 'lite' in n))
if MODELO.removeprefix('models/') not in disponibles:
    raise ValueError('El modelo configurado no aparece disponible. Revisa AI Studio y GEMINI_MODEL.')
mod.analyzer.MODEL_NAME = MODELO
mod.api.MODEL_NAME = MODELO
version = version_fuente(ENGINE, MODELO)
registro = Registro(version)
cliente = ClienteRegistrado(cliente_real, registro)
mod.analyzer.client = cliente
mod.api.client = cliente
print(json.dumps(version, ensure_ascii=False, indent=2))
print('Evidencia de esta ejecución:', registro.path)

Modelos Flash/Lite visibles: ['gemini-2.5-flash', 'gemini-2.5-flash-image', 'gemini-2.5-flash-lite', 'gemini-2.5-flash-native-audio-latest', 'gemini-2.5-flash-native-audio-preview-09-2025', 'gemini-2.5-flash-native-audio-preview-12-2025', 'gemini-2.5-flash-preview-tts', 'gemini-3-flash-preview', 'gemini-3.1-flash-image', 'gemini-3.1-flash-image-preview', 'gemini-3.1-flash-lite', 'gemini-3.1-flash-lite-image', 'gemini-3.1-flash-lite-preview', 'gemini-3.1-flash-live-preview', 'gemini-3.1-flash-tts-preview', 'gemini-3.5-flash', 'gemini-3.5-flash-lite', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'gemini-3.8-flash-lite-tts', 'gemini-3.8-flash-tts', 'gemini-flash-latest', 'gemini-flash-lite-latest', 'gemini-omni-1.1-flash', 'gemini-omni-flash-preview', 'veo-3.1-lite-generate-preview']
{
  "repositorio": "https://github.com/No-Country-simulation/G10-LATAM-EQUIPO-16",
  "pr": "https://github.com/No-Country-simulation/G10-LATAM-EQUIPO-16/pull/38",
  "commit": "d62dd02ce1a358824

## 3. Dataset y contrato de entrada

Se conservan los diez mensajes originales y se añaden tres candidatos en lotes independientes: dos logros y un feedback. Las expectativas describen la transformación buscada, sin fijar la clasificación del modelo ni exigir una redacción idéntica.

`tipo=mensaje` identifica la entrada; Gemini obtiene la clasificación semántica. Los JSON de AI Engine incluyen `id`; las variantes `_backend` se ajustan al DTO Java. Cada solicitud admite entre una y diez interacciones.


In [4]:
diagnostico = validar_lote(json.loads((BASE / 'datos/01_diagnostico_10.json').read_text(encoding='utf-8')))
expectativas = json.loads((BASE / 'datos/expectativas.json').read_text(encoding='utf-8'))
lotes_demo = {e['id']: validar_lote(json.loads((BASE / 'datos' / e['archivo']).read_text(encoding='utf-8'))) for e in expectativas}
import pandas as pd
display(pd.DataFrame(diagnostico['interacciones']).head(10))
display(pd.DataFrame(expectativas))
for caso, payload in lotes_demo.items():
    print(caso, json.dumps(payload, ensure_ascii=False, indent=2))
print('Contratos válidos: 10 mensajes de diagnóstico y 3 lotes individuales.')

,id,autor,canal,tipo,texto
0,MSG-001,Juan Perez,Discord,mensaje,¡Me encanta la nueva actualización! La interfa...
1,MSG-002,María Gómez,Slack,mensaje,El sistema no me deja exportar el PDF. Me tira...
2,MSG-003,Carlos Ruiz,Telegram,mensaje,¿Alguien sabe cómo se cambia el idioma del das...
3,MSG-004,Ana Martinez,Discord,mensaje,Sería increíble si pudieran agregar modo oscur...
4,MSG-005,Pedro Sánchez,Slack,mensaje,Me han cobrado dos veces la suscripción este m...
5,MSG-006,Laura Torres,Comunidad Web,mensaje,Gracias al equipo de soporte por ayudarme a mi...
6,MSG-007,Diego Lopez,Telegram,mensaje,El link para restablecer la contraseña me lleg...
7,MSG-008,Sofía Castro,Slack,mensaje,La app móvil se queda congelada cada vez que i...
8,MSG-009,Roberto Fernández,Discord,mensaje,¡Gana dinero rápido desde casa! Invierte en la...
9,MSG-010,Elena Vega,Comunidad Web,mensaje,¿Tienen documentación sobre cómo conectar esto...


,archivo,id,formato_esperado,proposito,aprobacion_marketing,ejecutado
0,02_demo_empleo.json,DEMO-001,linkedin,"Mostrar contratación sin inventar empresa, sal...",pendiente,False
1,03_demo_proyecto.json,DEMO-002,linkedin,Mostrar un logro técnico sin convertirlo en em...,pendiente,False
2,04_demo_feedback.json,DEMO-003,newsletter,Resumir feedback concreto sin prometer mejoras...,pendiente,False


DEMO-001 {
  "origen_comunidad": "Comunidad ficticia de prueba",
  "periodo_referencia": "Demo 2026-10-18",
  "interacciones": [
    {
      "id": "DEMO-001",
      "autor": "Alex Rivera",
      "canal": "Discord",
      "tipo": "mensaje",
      "texto": "Conseguí una pasantía de desarrollo frontend después de presentar mi proyecto de React."
    }
  ]
}
DEMO-002 {
  "origen_comunidad": "Comunidad ficticia de prueba",
  "periodo_referencia": "Demo 2026-10-18",
  "interacciones": [
    {
      "id": "DEMO-002",
      "autor": "Sam López",
      "canal": "Discord",
      "tipo": "mensaje",
      "texto": "Terminé mi proyecto de análisis de datos con Python y publiqué el repositorio para que puedan revisarlo."
    }
  ]
}
DEMO-003 {
  "origen_comunidad": "Comunidad ficticia de prueba",
  "periodo_referencia": "Demo 2026-10-18",
  "interacciones": [
    {
      "id": "DEMO-003",
      "autor": "Cris Vega",
      "canal": "Discord",
      "tipo": "mensaje",
      "texto": "Probé el nuevo pa

## 4. Evaluación de generadores

Tres análisis y hasta tres generaciones mediante las funciones originales. El formato se selecciona según la expectativa del caso para evaluar cada generador de forma aislada. Se registran análisis, prompts, respuestas y errores.

Esta etapa es opcional. La selección automática de ruta se evalúa en la etapa siguiente.


In [5]:
for exp in expectativas:
    caso = exp['id'] + '-directa'
    fila = lotes_demo[exp['id']]['interacciones'][0]
    interaction = mod.models.Interaction(**{k: v for k, v in fila.items() if k != 'id'})
    segura = mod.privacy.anonymize_interaction(interaction)
    cliente.contexto = caso
    registro.anotar('entrada', nivel_prueba='generador_directo', caso=caso, entrada=fila)
    try:
        analysis, metrics = mod.analyzer.analyze_text(segura.texto)
        funcion = (mod.generators.generate_linkedin if exp['formato_esperado'] == 'linkedin'
                   else mod.generators.generate_newsletter)
        salida = funcion(client=cliente, interaction=segura, analysis=analysis, model=MODELO)
        registro.anotar('resultado', nivel_prueba='generador_directo', caso=caso,
                       analisis=analysis.model_dump(), metricas=metrics, salida=salida)
        registro.anotar('activo', nivel_prueba='generador_directo', caso=caso,
                       activo=exp['formato_esperado'], entrada=fila, salida=salida)
        print('\nCASO:', caso, '\nANÁLISIS:', analysis.model_dump(), '\nRESPUESTA:', salida)
    except Exception as exc:
        registro.anotar('error', nivel_prueba='generador_directo', caso=caso, clase_error=type(exc).__name__)
        print(caso, 'ERROR:', type(exc).__name__)
    registro.exportar_revision()

🤖 Consultando gemini-3.5-flash-lite...


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.



--- USAGE DEBUG ---
cached_tokens_by_modality=None grounding_tool_count=None input_tokens_by_modality=[ModalityTokens(modality='text', tokens=281)] output_tokens_by_modality=None tool_use_tokens_by_modality=None total_cached_tokens=0 total_input_tokens=281 total_output_tokens=73 total_thought_tokens=0 total_tokens=354 total_tool_use_tokens=0

CASO: DEMO-001-directa 
ANÁLISIS: {'sentimiento': 'positivo', 'tema': 'empleabilidad', 'tipo': 'logro', 'relevancia': 'alta', 'insight': 'Los proyectos prácticos impulsan directamente la inserción laboral de los miembros, lo cual puede usarse como caso de éxito en marketing.'} 
RESPUESTA: ¡La práctica se convierte en oportunidades!

Una persona de nuestra comunidad consiguió una pasantía de desarrollo frontend después de presentar su proyecto de React. 

Construir proyectos reales es el camino directo para demostrar habilidades técnicas y abrir puertas en el sector tecnológico. ¡Muchas felicidades por este gran paso!

Únete a nuestra comunidad en

## 5. Evaluación del AI Engine local

Ejecución de `/api/v1/analyze-batch` mediante TestClient, con análisis y generación reales en Gemini. Incluye selección por relevancia, un activo por tipo y lote, y enrutamiento mediante LangGraph.

Se procesan tres lotes individuales. Newsletter requiere clasificación `feedback` y relevancia `alta`. La ruta obtenida y sus respuestas se conservan para la curación del dataset.

Cobertura: AI Engine local. Integraciones Java, OCI y filtro MiniLM pendientes de conexión en este snapshot.


In [6]:
resultados_locales = {}
for caso, payload in lotes_demo.items():
    status, salida = probar_motor(mod, cliente, registro, caso, payload)
    resultados_locales[caso] = {'http_status': status, 'salida': salida}
    print('\nCASO:', caso, 'HTTP:', status)
    print(json.dumps(salida, ensure_ascii=False, indent=2))
guardar_json(registro.path / 'resultados_ai_engine.json', resultados_locales)

🤖 Analizando lote de 1 interacciones con gemini-3.5-flash-lite...

CASO: DEMO-001 HTTP: 200
{
  "status": "exito",
  "resumen_comunidad": {
    "total_interacciones_procesadas": 1,
    "sentimiento_predominante": "positivo",
    "temas_principales": [
      "Empleabilidad en desarrollo frontend"
    ]
  },
  "activos_distribucion_generados": {
    "post_linkedin": {
      "interaccion_id": "DEMO-001",
      "titulo": "Empleabilidad en desarrollo frontend",
      "copy": "¡Los proyectos prácticos abren puertas!\n\nUna persona de nuestra comunidad consiguió una pasantía de desarrollo frontend después de presentar su proyecto de React. Este logro demuestra cómo la práctica directa en desarrollo frontend es efectiva para conseguir oportunidades en el sector.\n\nConstruir y presentar proyectos marca la diferencia en el camino profesional. ¡Muchas felicidades por este gran paso!\n\nÚnete a nuestra comunidad en Discord.\n\n#DesarrolloFrontend #React #EmpleabilidadTech #ComunidadDev",
      "c

## 6. Diagnóstico de los diez mensajes originales

Evaluación opcional de consultas, quejas, feedback y spam. Puede producir FAQ. El enrutador selecciona newsletter para feedback de relevancia alta, independientemente del sentimiento; esta condición se incluye en la revisión de resultados.


In [7]:
EJECUTAR_DIAGNOSTICO = False  # cambiar a True para hacer esta solicitud adicional
if EJECUTAR_DIAGNOSTICO:
    status, salida = probar_motor(mod, cliente, registro, 'DIAGNOSTICO-10', diagnostico)
    print('HTTP:', status)
    print(json.dumps(salida, ensure_ascii=False, indent=2))
else:
    print('Diagnóstico no ejecutado; no se registra como prueba realizada.')

Diagnóstico no ejecutado; no se registra como prueba realizada.


## 7. Evaluación mediante el backend integrado

Solicitud HTTP al entorno de prueba. Configuración: `BACKEND_URL`, `BACKEND_VERSION` y, cuando corresponda, `BACKEND_TOKEN`.

`BACKEND_VERSION` identifica las versiones desplegadas de Java y AI Engine y el modelo configurado en el servidor. La versión es declarada por el operador. Las entradas se ajustan al contrato mediante los archivos `*_backend.json`.

Esta ejecución puede persistir registros y objetos en las integraciones del entorno.


In [9]:
EJECUTAR_BACKEND = False  # activar cuando el entorno de pruebas esté disponible
if EJECUTAR_BACKEND:
    load_dotenv(BASE / '.env', override=True)
    resultados_backend = {}
    for caso, payload in lotes_demo.items():
        status, salida = probar_backend(registro, caso, payload,
            os.environ.get('BACKEND_URL', ''), os.environ.get('BACKEND_VERSION', ''),
            os.environ.get('BACKEND_TOKEN', ''))
        resultados_backend[caso] = {'http_status': status, 'salida': salida}
        print('\nCASO:', caso, 'HTTP:', status)
        print(json.dumps(salida, ensure_ascii=False, indent=2))
    guardar_json(registro.path / 'resultados_backend.json', resultados_backend)
else:
    print('Backend no ejecutado. Las pruebas locales no se presentan como ejecución en OCI.')

Backend no ejecutado. Las pruebas locales no se presentan como ejecución en OCI.


## 8. Evidencia y revisión editorial

`evidencia.jsonl` registra por evento entradas, prompts, respuestas, resultados HTTP y errores. `version_probada.json` identifica fuente, modelo y dependencias. `revision_marketing.csv` reúne los activos y los campos de evaluación editorial.

Criterios: fidelidad, anonimización, tratamiento de género, tono y formato. Los niveles de prueba se mantienen identificados para distinguir resultados del generador, AI Engine y backend.


In [10]:
registro.exportar_revision()
print('Archivos persistidos:', [p.name for p in registro.path.iterdir()])
activos = [r for r in registro.filas if r['tipo_registro'] == 'activo']
if activos:
    display(pd.DataFrame(activos)[['fecha_utc', 'nivel_prueba', 'caso', 'activo', 'salida']])
    for nivel in sorted({r['nivel_prueba'] for r in activos}):
        casos = {r['caso'] for r in activos if r['nivel_prueba'] == nivel}
        print(nivel, ':', len(casos), 'casos con activo. Revisión editorial pendiente.')
else:
    print('Todavía no hay activos obtenidos; no hay evidencia de tres transformaciones.')

Archivos persistidos: ['evidencia.jsonl', 'resultados_ai_engine.json', 'revision_marketing.csv', 'version_probada.json']


,fecha_utc,nivel_prueba,caso,activo,salida
0,2026-10-18T03:01:56.745089+00:00,generador_directo,DEMO-001-directa,linkedin,¡La práctica se convierte en oportunidades!\n\...
1,2026-10-18T03:01:59.286096+00:00,generador_directo,DEMO-002-directa,linkedin,El aprendizaje práctico se ve reflejado en cad...
2,2026-10-18T03:02:01.162628+00:00,generador_directo,DEMO-003-directa,newsletter,Una persona de la comunidad probó el nuevo pan...
3,2026-10-18T03:02:08.581908+00:00,ai_engine_local,DEMO-001,post_linkedin,"{'interaccion_id': 'DEMO-001', 'titulo': 'Empl..."
4,2026-10-18T03:02:10.472044+00:00,ai_engine_local,DEMO-002,post_linkedin,"{'interaccion_id': 'DEMO-002', 'titulo': 'Proy..."
5,2026-10-18T03:02:12.458824+00:00,ai_engine_local,DEMO-003,post_linkedin,"{'interaccion_id': 'DEMO-003', 'titulo': 'Pane..."


ai_engine_local : 3 casos con activo. Revisión editorial pendiente.
generador_directo : 3 casos con activo. Revisión editorial pendiente.


## 9. Entrega de curación

Artefactos: notebook ejecutado, `curacion.py`, entradas CSV/JSON, expectativas, requisitos, documentación y evidencia seleccionada.

El criterio de cierre requiere al menos tres transformaciones identificables del flujo integrado, con entrada, salida, fecha y versión, disponibles para revisión editorial y preparación de la demo.


In [11]:
activos[0]

{'fecha_utc': '2026-10-18T03:01:56.745089+00:00',
 'tipo_registro': 'activo',
 'nivel_prueba': 'generador_directo',
 'caso': 'DEMO-001-directa',
 'activo': 'linkedin',
 'entrada': {'id': 'DEMO-001',
  'autor': 'Alex Rivera',
  'canal': 'Discord',
  'tipo': 'mensaje',
  'texto': 'Conseguí una pasantía de desarrollo frontend después de presentar mi proyecto de React.'},
 'salida': '¡La práctica se convierte en oportunidades!\n\nUna persona de nuestra comunidad consiguió una pasantía de desarrollo frontend después de presentar su proyecto de React. \n\nConstruir proyectos reales es el camino directo para demostrar habilidades técnicas y abrir puertas en el sector tecnológico. ¡Muchas felicidades por este gran paso!\n\nÚnete a nuestra comunidad en Discord.\n\n#Frontend #DesarrolloWeb #EmpleabilidadTech'}

In [12]:
print(activos[0]['salida'])

¡La práctica se convierte en oportunidades!

Una persona de nuestra comunidad consiguió una pasantía de desarrollo frontend después de presentar su proyecto de React. 

Construir proyectos reales es el camino directo para demostrar habilidades técnicas y abrir puertas en el sector tecnológico. ¡Muchas felicidades por este gran paso!

Únete a nuestra comunidad en Discord.

#Frontend #DesarrolloWeb #EmpleabilidadTech
